<a href="https://colab.research.google.com/github/ykhier/Arduino_calculator_project/blob/main/lab2___studentForm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [76]:
import json
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [77]:
students = [
    {
        "first_name": "Yosef",
        "last_name": "Khier",
        "braude_email": "yosef.khier@braude.ac.il",
        "courses": ["Algorithms", "Operating Systems", "Databases"],
        "interesting_link": "https://chatgpt.com/",
        "favorite_program": ""
    },
    {
        "first_name": "Lama",
        "last_name": "Hassan",
        "braude_email": "lama.abuabla@braude.ac.il",
        "courses": ["Discrete Mathematics", "Data Structures", "Computer Networks"],
        "interesting_link": "https://www.sport5.co.il/",
        "favorite_program": ""
    }
]

path = "/content/drive/My Drive/students.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(students, f, ensure_ascii=False, indent=2)


In [78]:

with open("/content/drive/My Drive/students.json", "r", encoding="utf-8") as fid:
    jsonData = json.load(fid)
print(jsonData)


[{'first_name': 'Yosef', 'last_name': 'Khier', 'braude_email': 'yosef.khier@braude.ac.il', 'courses': ['Algorithms', 'Operating Systems', 'Databases'], 'interesting_link': 'https://chatgpt.com/', 'favorite_program': ''}, {'first_name': 'Lama', 'last_name': 'Hassan', 'braude_email': 'lama.abuabla@braude.ac.il', 'courses': ['Discrete Mathematics', 'Data Structures', 'Computer Networks'], 'interesting_link': 'https://www.sport5.co.il/', 'favorite_program': ''}]


In [79]:
import ipywidgets as widgets
from IPython.display import display

def getStudents():
  with open('/content/drive/My Drive/students.json', 'r', encoding='utf-8') as file:
      studentsData = json.load(file)
  return studentsData



In [80]:
print(getStudents()[0])

{'first_name': 'Yosef', 'last_name': 'Khier', 'braude_email': 'yosef.khier@braude.ac.il', 'courses': ['Algorithms', 'Operating Systems', 'Databases'], 'interesting_link': 'https://chatgpt.com/', 'favorite_program': ''}


In [81]:
def populate_fields(idx):
        s = students[idx]
        email_tb.value   = s.get("braude_email")
        coursesStr = ""
        for course in s.get("courses"):
            coursesStr += course
        courses_tb.value = coursesStr
        link_tb.value    = s.get("interesting_link")

In [82]:
def on_student_change(change):
        idx = names.index(change["new"])
        populate_fields(idx)

In [83]:
def on_append_clicked(b):
    idx = names.index(studentField.value)
    student = students[idx]
    fav = fav_tb.value.strip()
    student["favorite_program"] = fav

    with open("/content/drive/My Drive/students.json", "w", encoding="utf-8") as f:
        json.dump(students, f, ensure_ascii=False, indent=2)

    fav_tb.value = ""
    populate_fields(idx)

In [84]:


students = getStudents()
names = []

for student in students:
  names.append(f"{student["first_name"]} {student["last_name"]}")

studentField = widgets.Dropdown(options=names, description='סטודנט:', layout=widgets.Layout(width='400px'))
email_tb   = widgets.Text(description='מייל:',    disabled=True, layout=widgets.Layout(width='600px'))
courses_tb = widgets.Textarea(description='קורסים:', disabled=True, layout=widgets.Layout(width='600px', height='80px'))
link_tb    = widgets.Text(description='קישור:',   disabled=True, layout=widgets.Layout(width='600px'))
fav_tb     = widgets.Text(description='תוכנית אהובה:', placeholder='לדוגמה: Python', layout=widgets.Layout(width='400px'))
append_btn = widgets.Button(description='הוסף שורה לקובץ', button_style='success')

populate_fields(0)

form = widgets.VBox([
    studentField,
    email_tb,
    courses_tb,
    link_tb,
    widgets.HBox([fav_tb, append_btn])
])
display(form)

studentField.observe(on_student_change, names="value")

append_btn.on_click(on_append_clicked)




In [86]:
!pip -q install nbformat

import os, nbformat
from nbformat import read, write
from nbformat.validator import validate

REPO = "/content/Cloud_Course"
PATH = f"{REPO}/lab2_studentForm.ipynb"

# Clone אם צריך
if not os.path.exists(REPO):
    !git clone https://github.com/ykhier/Cloud_Course.git

# --- ניקוי עמוק של מטא-דאטה ופלטים ---
nb = read(PATH, as_version=4)

# מחיקת metadata.widgets בשורש
nb.metadata.pop("widgets", None)

# ניקוי תאים
for cell in nb.cells:
    # הסר מטא-דאטה של widgets בתא
    cell.metadata.pop("widgets", None)
    # נקה פלטים + MIME של וידג׳טים
    if "outputs" in cell:
        new_outputs = []
        for out in cell["outputs"]:
            if isinstance(out, dict) and "data" in out and isinstance(out["data"], dict):
                out["data"].pop("application/vnd.jupyter.widget-view+json", None)
                out["data"].pop("application/vnd.jupyter.widget-state+json", None)
            # אפשר גם לאפס הכל:
            # continue  # לדלג על כל הפלטים ולמחוק הכל
            new_outputs.append(out)
        # אם רוצים למחוק את כל הפלטים לגמרי:
        new_outputs = []
        cell["outputs"] = new_outputs
    if "execution_count" in cell:
        cell["execution_count"] = None

# ודא kernelspec בסיסי
nb.metadata.setdefault("kernelspec", {"display_name":"Python 3", "language":"python", "name":"python3"})
nb.metadata.setdefault("language_info", {"name":"python"})

# כתיבה
write(nb, PATH)

# ולידציה (יקרוס אם לא תקין)
validate(nb)

# --- commit + push ---
%cd $REPO
!git config user.email "you@example.com"
!git config user.name "yourname"
!git add lab2_studentForm.ipynb
!git commit -m "Strip widgets metadata/outputs from lab2_studentForm.ipynb"
!git push origin main


/content/Cloud_Course
[main 16d7117] Strip widgets metadata/outputs from lab2_studentForm.ipynb
 1 file changed, 252 insertions(+), 987 deletions(-)
 rewrite lab2_studentForm.ipynb (99%)
fatal: could not read Username for 'https://github.com': No such device or address
